# LeetCode 1578 — Minimum Time to Make Rope Colorful bằng Python

Link đề: https://leetcode.com/problems/minimum-time-to-make-rope-colorful/ · Độ khó: **Medium** · Chủ đề: Array, String, Dynamic Programming, Greedy

Skeleton LeetCode:

```python
class Solution:
    def minCost(self, colors: str, neededTime: List[int]) -> int:
        ...
```

## 1. Mục tiêu học

1. Nhận ra rằng chỉ các **bóng cùng màu đứng liền nhau** mới gây xung đột, nên bài toán tách thành các **nhóm liên tiếp cùng màu** độc lập.
2. Viết baseline **thử mọi tập bóng bị gỡ** (`2ⁿ`) để có chuẩn đối chiếu.
3. Lập **quy hoạch động** theo "màu của bóng cuối cùng được giữ" — cách tổng quát áp dụng được cho nhiều biến thể.
4. Rút ra công thức tham lam: mỗi nhóm giữ lại **đúng một** bóng có thời gian lớn nhất, chi phí = `tổng - max`; và viết bản **một lượt** chỉ giữ biến `max` hiện tại.
5. Tránh các bẫy: chỉ so sánh cặp kề nhau mà không mang theo `max`, gom theo màu toàn cục thay vì theo nhóm liên tiếp, quên nhóm cuối.

## 2. Đề bài & ví dụ

Alice có `n` quả bóng xếp trên một sợi dây. Chuỗi `colors` cho biết `colors[i]` là màu của quả bóng thứ `i`.

Alice muốn sợi dây **nhiều màu**: **không có hai quả bóng liền nhau cùng màu**. Bob có thể gỡ bỏ một số quả bóng; gỡ quả thứ `i` mất `neededTime[i]` giây.

Trả về **thời gian nhỏ nhất** để sợi dây trở nên nhiều màu.

**Ví dụ 1:** `colors = "abaac"`, `neededTime = [1,2,3,4,5]` → `3`. Gỡ bóng ở chỉ số 2 (màu `a`, 3 giây).

**Ví dụ 2:** `colors = "abc"`, `neededTime = [1,2,3]` → `0`. Đã nhiều màu sẵn.

**Ví dụ 3:** `colors = "aabaa"`, `neededTime = [1,2,3,4,1]` → `2`. Gỡ bóng ở chỉ số 0 và 4.

**Ràng buộc:**
- `n == colors.length == neededTime.length`
- `1 ≤ n ≤ 10⁵`
- `1 ≤ neededTime[i] ≤ 10⁴`
- `colors` chỉ gồm chữ cái tiếng Anh thường.

Lưu ý: sau khi gỡ, các bóng còn lại **dồn lại** — nên "liền nhau" được xét trên dãy còn lại.

In [1]:
import random
import time
from itertools import groupby
from typing import List

examples = [("abaac", [1, 2, 3, 4, 5], 3), ("abc", [1, 2, 3], 0), ("aabaa", [1, 2, 3, 4, 1], 2)]
for colors, t, want in examples:
    print(f"colors={colors!r}, neededTime={t} -> mong đợi {want}")

colors='abaac', neededTime=[1, 2, 3, 4, 5] -> mong đợi 3
colors='abc', neededTime=[1, 2, 3] -> mong đợi 0
colors='aabaa', neededTime=[1, 2, 3, 4, 1] -> mong đợi 2


## 3. Kiến thức nền

### 3.1 `zip` hai dãy song song

Duyệt màu và thời gian cùng lúc: `for c, t in zip(colors, neededTime)`.

In [2]:
colors, needed = "abaac", [1, 2, 3, 4, 5]
for i, (c, t) in enumerate(zip(colors, needed)):
    print(f"bóng {i}: màu {c}, thời gian {t}")

bóng 0: màu a, thời gian 1
bóng 1: màu b, thời gian 2
bóng 2: màu a, thời gian 3
bóng 3: màu a, thời gian 4
bóng 4: màu c, thời gian 5


### 3.2 `itertools.groupby` — gom các phần tử **liên tiếp** bằng nhau

`groupby(iterable, key)` tạo các nhóm **liền nhau** có cùng `key`. Khác với "gom theo giá trị" toàn cục: `"aabaa"` cho 3 nhóm `aa`, `b`, `aa`.

In [3]:
print([(k, "".join(g)) for k, g in groupby("aabaa")])
pairs = list(zip("aabaa", [1, 2, 3, 4, 1]))
for color, grp in groupby(pairs, key=lambda p: p[0]):
    times = [t for _, t in grp]
    print(f"nhóm màu {color}: thời gian {times}, tổng {sum(times)}, max {max(times)}")

[('a', 'aa'), ('b', 'b'), ('a', 'aa')]
nhóm màu a: thời gian [1, 2], tổng 3, max 2
nhóm màu b: thời gian [3], tổng 3, max 3
nhóm màu a: thời gian [4, 1], tổng 5, max 4


### 3.3 Duyệt mọi tập con bằng bitmask

Với `n` nhỏ, mỗi số `mask` từ `0` đến `2ⁿ - 1` biểu diễn một tập con: bit `i` bật ⇔ phần tử `i` thuộc tập. Ta dùng để thử mọi cách gỡ bóng.

In [4]:
n = 3
for mask in range(1 << n):
    chosen = [i for i in range(n) if mask >> i & 1]
    print(f"mask={mask:03b} -> tập {chosen}")

mask=000 -> tập []
mask=001 -> tập [0]
mask=010 -> tập [1]
mask=011 -> tập [0, 1]
mask=100 -> tập [2]
mask=101 -> tập [0, 2]
mask=110 -> tập [1, 2]
mask=111 -> tập [0, 1, 2]


### 3.4 `min`/`max` và cập nhật "giá trị lớn nhất đang chạy"

Mẫu `cur_max = max(cur_max, x)` giữ phần tử lớn nhất đã thấy. Kết hợp `ans += min(cur_max, x)` cho phép "gỡ cái nhỏ hơn, giữ cái lớn hơn" trong một lượt.

In [5]:
times = [3, 1, 2]
ans, cur_max = 0, times[0]
for x in times[1:]:
    removed = min(cur_max, x)
    ans += removed
    cur_max = max(cur_max, x)
    print(f"x={x}: gỡ phần tử giá {removed}, ans={ans}, giữ max={cur_max}")
print("tổng - max =", sum(times) - max(times))

x=1: gỡ phần tử giá 1, ans=1, giữ max=3
x=2: gỡ phần tử giá 2, ans=3, giữ max=3
tổng - max = 3


### 3.5 Quy hoạch động với từ điển trạng thái

Ta có thể lưu trạng thái dạng `dict` (khoá = màu, giá trị = chi phí nhỏ nhất). Mỗi bước tạo `dict` mới từ `dict` cũ. Với 26 màu + trạng thái "chưa giữ quả nào", mỗi bước tốn `O(27)`.

In [6]:
INF = float("inf")
dp = {None: 0}            # chưa giữ quả nào, chi phí 0
print("dp ban đầu:", dp)
dp2 = {k: v + 5 for k, v in dp.items()}   # ví dụ: gỡ một quả tốn 5
dp2["a"] = min(dp2.get("a", INF), dp[None])  # hoặc giữ quả màu 'a'
print("sau một bước:", dp2)

dp ban đầu: {None: 0}
sau một bước: {None: 5, 'a': 0}


## 4. Ý tưởng nền tảng / trực giác

Ví dụ 3: `colors = "aabaa"`, `neededTime = [1,2,3,4,1]`.

Các nhóm liên tiếp cùng màu: `aa (1,2)` | `b (3)` | `aa (4,1)`.

- Hai nhóm kề nhau luôn **khác màu** (nếu không thì đã là một nhóm), nên chỉ cần mỗi nhóm còn **tối đa 1** quả. Nhóm phải còn **ít nhất 1** quả? Không bắt buộc, nhưng gỡ hết thì tốn thêm mà không lợi gì — và có thể làm hai nhóm cùng màu ở hai bên dồn lại gần nhau! Vì vậy giữ đúng 1 quả là tối ưu.
- Giữ quả **tốn thời gian gỡ nhất**, gỡ các quả còn lại:
  - nhóm `aa (1,2)`: giữ 2, gỡ 1 → chi phí 1;
  - nhóm `b (3)`: chi phí 0;
  - nhóm `aa (4,1)`: giữ 4, gỡ 1 → chi phí 1.
- Tổng = `2`. ✔

Ví dụ 1: `"abaac"` → nhóm `a(1) | b(2) | aa(3,4) | c(5)` → chỉ nhóm `aa` mất `3 + 4 - 4 = 3`. ✔

In [7]:
def show_groups(colors, needed):
    total = 0
    for color, grp in groupby(zip(colors, needed), key=lambda p: p[0]):
        ts = [t for _, t in grp]
        cost = sum(ts) - max(ts)
        total += cost
        print(f"  nhóm {color * len(ts):<5} thời gian {ts}: giữ {max(ts)}, gỡ tốn {cost}")
    print("  tổng =", total)

for colors, t, _ in examples:
    print(colors, t)
    show_groups(colors, t)

abaac [1, 2, 3, 4, 5]
  nhóm a     thời gian [1]: giữ 1, gỡ tốn 0
  nhóm b     thời gian [2]: giữ 2, gỡ tốn 0
  nhóm aa    thời gian [3, 4]: giữ 4, gỡ tốn 3
  nhóm c     thời gian [5]: giữ 5, gỡ tốn 0
  tổng = 3
abc [1, 2, 3]
  nhóm a     thời gian [1]: giữ 1, gỡ tốn 0
  nhóm b     thời gian [2]: giữ 2, gỡ tốn 0
  nhóm c     thời gian [3]: giữ 3, gỡ tốn 0
  tổng = 0
aabaa [1, 2, 3, 4, 1]
  nhóm aa    thời gian [1, 2]: giữ 2, gỡ tốn 1
  nhóm b     thời gian [3]: giữ 3, gỡ tốn 0
  nhóm aa    thời gian [4, 1]: giữ 4, gỡ tốn 1
  tổng = 2


## 5. Cách 1 — Brute Force / baseline

**Ý tưởng:** thử mọi tập con bóng bị gỡ (bitmask). Với mỗi tập, lấy dãy màu còn lại, kiểm tra không có hai màu liền nhau trùng; nếu hợp lệ thì cập nhật chi phí nhỏ nhất.

In [8]:
def brute_force(colors: str, neededTime: List[int]) -> int:
    n = len(colors)
    best = sum(neededTime)                   # gỡ hết luôn hợp lệ
    for mask in range(1 << n):               # bit i bật = gỡ bóng i
        kept = [colors[i] for i in range(n) if not mask >> i & 1]
        if all(a != b for a, b in zip(kept, kept[1:])):
            cost = sum(neededTime[i] for i in range(n) if mask >> i & 1)
            best = min(best, cost)
    return best

for colors, t, want in examples:
    print(colors, t, "->", brute_force(colors, t), "| mong đợi", want)

abaac [1, 2, 3, 4, 5] -> 3 | mong đợi 3
abc [1, 2, 3] -> 0 | mong đợi 0
aabaa [1, 2, 3, 4, 1] -> 2 | mong đợi 2


**Độ phức tạp:** `O(2ⁿ · n)` thời gian, `O(n)` bộ nhớ. Chỉ dùng được với `n ≤ ~16`.

**Nút thắt:** số tập con bùng nổ theo cấp số mũ. Nhưng thực ra quyết định "gỡ/giữ" một quả chỉ phụ thuộc vào **màu của quả cuối cùng được giữ** trước nó — gợi ý quy hoạch động.

In [9]:
rng = random.Random(1)
colors_big = "".join(rng.choice("ab") for _ in range(16))
t_big = [rng.randint(1, 10) for _ in range(16)]
t0 = time.perf_counter()
r = brute_force(colors_big, t_big)
print(f"n=16: {r}, mất {time.perf_counter() - t0:.3f}s -> n=30 sẽ chậm gấp ~16000 lần")

n=16: 9, mất 0.087s -> n=30 sẽ chậm gấp ~16000 lần


## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 (tách nhóm):** Xung đột chỉ xảy ra giữa hai quả **cùng màu** mà giữa chúng **không còn quả khác màu**. Nếu ta giữ ít nhất một quả ở mỗi nhóm thì các quả khác màu ở giữa luôn ngăn cách các nhóm ⇒ các nhóm **độc lập**.

**Insight 2 (mỗi nhóm giữ đúng 1):** Trong một nhóm cùng màu liên tiếp, giữ ≥ 2 quả là vi phạm. Giữ 0 quả không bao giờ tốt hơn giữ 1 quả (tốn thêm thời gian, và có thể khiến hai nhóm cùng màu hai bên áp sát nhau). ⇒ giữ đúng 1 quả.

**Insight 3 (giữ quả đắt nhất):** chi phí nhóm = `tổng - (thời gian của quả được giữ)` ⇒ nhỏ nhất khi giữ quả có thời gian **lớn nhất**:

`đáp án = Σ_nhóm (tổng nhóm − max nhóm)`

**Quy tắc một lượt:** *nếu* `colors[i] == colors[i-1]` *thì* gỡ quả nhỏ hơn giữa `cur_max` và `neededTime[i]` (cộng `min`), rồi `cur_max = max(...)`; *ngược lại* bắt đầu nhóm mới: `cur_max = neededTime[i]`.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Quy hoạch động theo màu của quả cuối được giữ

**Trạng thái:** `dp[c]` = chi phí nhỏ nhất sau khi xử lý các quả `0..i`, với **quả cuối cùng được giữ có màu `c`** (`c = None` nếu chưa giữ quả nào).

**Chuyển trạng thái** với quả `i` (màu `x`, thời gian `t`):
- **gỡ** quả `i`: mọi trạng thái giữ nguyên màu, cộng `t`: `new[c] = dp[c] + t`;
- **giữ** quả `i`: chỉ được nếu quả cuối được giữ trước đó **khác màu** `x`: `new[x] = min(new[x], min_{c ≠ x} dp[c])`.

**Đáp án:** `min(dp.values())`.

Đây là cách **tổng quát** — không cần nhìn ra insight về nhóm, và áp dụng được cho các biến thể ràng buộc phức tạp hơn.

In [10]:
def min_cost_dp(colors: str, neededTime: List[int]) -> int:
    dp = {None: 0}
    for x, t in zip(colors, neededTime):
        new = {c: v + t for c, v in dp.items()}              # gỡ quả hiện tại
        keep = min((v for c, v in dp.items() if c != x), default=float("inf"))
        if keep < new.get(x, float("inf")):
            new[x] = keep                                    # giữ quả hiện tại
        dp = new
    return min(dp.values())

for colors, t, want in examples:
    print(colors, t, "->", min_cost_dp(colors, t), "| mong đợi", want)

abaac [1, 2, 3, 4, 5] -> 3 | mong đợi 3
abc [1, 2, 3] -> 0 | mong đợi 0
aabaa [1, 2, 3, 4, 1] -> 2 | mong đợi 2


Code nộp LeetCode cho cách 2:

```python
class Solution:
    def minCost(self, colors: str, neededTime: List[int]) -> int:
        dp = {None: 0}
        for x, t in zip(colors, neededTime):
            new = {c: v + t for c, v in dp.items()}
            keep = min((v for c, v in dp.items() if c != x), default=float("inf"))
            if keep < new.get(x, float("inf")):
                new[x] = keep
            dp = new
        return min(dp.values())
```

**Độ phức tạp:** `O(n · Σ)` thời gian với `Σ ≤ 27` trạng thái, `O(Σ)` bộ nhớ. Chạy được với `n = 10⁵` nhưng chậm hơn cách tham lam khoảng vài chục lần.

### 7.2 Cách 3 — Nhóm liên tiếp: `tổng − max` mỗi nhóm

**Đoạn 1:** dùng hai con trỏ `i`, `j` (hoặc `groupby`) để tìm nhóm `colors[i..j-1]` cùng màu.

**Đoạn 2:** với mỗi nhóm, cộng `sum(neededTime[i:j]) - max(neededTime[i:j])` (nhóm 1 phần tử cho 0).

**Đoạn 3:** nhảy `i = j` sang nhóm tiếp theo.

In [11]:
def min_cost_groups(colors: str, neededTime: List[int]) -> int:
    n, ans, i = len(colors), 0, 0
    while i < n:
        j = i
        s = mx = 0
        while j < n and colors[j] == colors[i]:
            s += neededTime[j]
            mx = max(mx, neededTime[j])
            j += 1
        ans += s - mx
        i = j
    return ans

def min_cost_groupby(colors: str, neededTime: List[int]) -> int:
    ans = 0
    for _, grp in groupby(zip(colors, neededTime), key=lambda p: p[0]):
        ts = [t for _, t in grp]
        ans += sum(ts) - max(ts)
    return ans

for colors, t, want in examples:
    print(colors, t, "->", min_cost_groups(colors, t), min_cost_groupby(colors, t), "| mong đợi", want)

abaac [1, 2, 3, 4, 5] -> 3 3 | mong đợi 3
abc [1, 2, 3] -> 0 0 | mong đợi 0
aabaa [1, 2, 3, 4, 1] -> 2 2 | mong đợi 2


Code nộp LeetCode cho cách 3:

```python
class Solution:
    def minCost(self, colors: str, neededTime: List[int]) -> int:
        n, ans, i = len(colors), 0, 0
        while i < n:
            j, s, mx = i, 0, 0
            while j < n and colors[j] == colors[i]:
                s += neededTime[j]
                mx = max(mx, neededTime[j])
                j += 1
            ans += s - mx
            i = j
        return ans
```

**Độ phức tạp:** `O(n)` thời gian (mỗi chỉ số được `j` đi qua đúng một lần), `O(1)` bộ nhớ phụ.

### 7.3 Cách 4 — Một lượt tham lam, giữ `max` của nhóm hiện tại

Không cần tách nhóm tường minh. Duyệt từ trái sang phải với biến `cur_max` = thời gian lớn nhất của nhóm hiện tại (chính là quả "đang được giữ"):
- nếu `colors[i] == colors[i-1]`: hai quả cùng màu kề nhau, phải gỡ một trong hai ⇒ gỡ quả rẻ hơn: `ans += min(cur_max, t)`, giữ quả đắt hơn: `cur_max = max(cur_max, t)`;
- ngược lại: nhóm mới, `cur_max = t`.

Tổng các `min` trong một nhóm đúng bằng `tổng − max` của nhóm đó.

In [12]:
def min_cost_greedy(colors: str, neededTime: List[int]) -> int:
    ans = 0
    cur_max = neededTime[0]
    for i in range(1, len(colors)):
        t = neededTime[i]
        if colors[i] == colors[i - 1]:
            ans += min(cur_max, t)
            cur_max = max(cur_max, t)
        else:
            cur_max = t
    return ans

for colors, t, want in examples:
    print(colors, t, "->", min_cost_greedy(colors, t), "| mong đợi", want)

abaac [1, 2, 3, 4, 5] -> 3 | mong đợi 3
abc [1, 2, 3] -> 0 | mong đợi 0
aabaa [1, 2, 3, 4, 1] -> 2 | mong đợi 2


Code nộp LeetCode cho cách 4 (giống phần 9):

```python
class Solution:
    def minCost(self, colors: str, neededTime: List[int]) -> int:
        ans, cur_max = 0, neededTime[0]
        for i in range(1, len(colors)):
            t = neededTime[i]
            if colors[i] == colors[i - 1]:
                ans += min(cur_max, t)
                cur_max = max(cur_max, t)
            else:
                cur_max = t
        return ans
```

**Độ phức tạp:** `O(n)` thời gian, `O(1)` bộ nhớ phụ. Đây là lời giải ngắn và nhanh nhất.

## 8. TRACE từng bước

Cách 4 với `colors = "aaabbbabbbb"`, `neededTime = [3,5,10,7,5,3,5,5,4,8,1]`.

| i | màu | t | cùng màu trước? | gỡ (min) | cur_max | ans |
|---|---|---|---|---|---|---|
| 0 | a | 3 | — | — | 3 | 0 |
| 1 | a | 5 | ✔ | 3 | 5 | 3 |
| 2 | a | 10 | ✔ | 5 | 10 | 8 |
| 3 | b | 7 | ✘ | — | 7 | 8 |
| 4 | b | 5 | ✔ | 5 | 7 | 13 |
| 5 | b | 3 | ✔ | 3 | 7 | 16 |
| 6 | a | 5 | ✘ | — | 5 | 16 |
| 7 | b | 5 | ✘ | — | 5 | 16 |
| 8 | b | 4 | ✔ | 4 | 5 | 20 |
| 9 | b | 8 | ✔ | 5 | 8 | 25 |
| 10 | b | 1 | ✔ | 1 | 8 | **26** |

In [13]:
def trace_greedy(colors, neededTime):
    ans, cur_max = 0, neededTime[0]
    print(f"i= 0 màu={colors[0]} t={neededTime[0]:>2} | bắt đầu nhóm   | cur_max={cur_max:>2} ans={ans}")
    for i in range(1, len(colors)):
        t = neededTime[i]
        if colors[i] == colors[i - 1]:
            removed = min(cur_max, t)
            ans += removed
            cur_max = max(cur_max, t)
            act = f"gỡ quả giá {removed:>2}"
        else:
            cur_max = t
            act = "nhóm mới      "
        print(f"i={i:>2} màu={colors[i]} t={t:>2} | {act} | cur_max={cur_max:>2} ans={ans}")
    return ans

print("kết quả:", trace_greedy("aaabbbabbbb", [3, 5, 10, 7, 5, 3, 5, 5, 4, 8, 1]))

i= 0 màu=a t= 3 | bắt đầu nhóm   | cur_max= 3 ans=0
i= 1 màu=a t= 5 | gỡ quả giá  3 | cur_max= 5 ans=3
i= 2 màu=a t=10 | gỡ quả giá  5 | cur_max=10 ans=8
i= 3 màu=b t= 7 | nhóm mới       | cur_max= 7 ans=8
i= 4 màu=b t= 5 | gỡ quả giá  5 | cur_max= 7 ans=13
i= 5 màu=b t= 3 | gỡ quả giá  3 | cur_max= 7 ans=16
i= 6 màu=a t= 5 | nhóm mới       | cur_max= 5 ans=16
i= 7 màu=b t= 5 | nhóm mới       | cur_max= 5 ans=16
i= 8 màu=b t= 4 | gỡ quả giá  4 | cur_max= 5 ans=20
i= 9 màu=b t= 8 | gỡ quả giá  5 | cur_max= 8 ans=25
i=10 màu=b t= 1 | gỡ quả giá  1 | cur_max= 8 ans=26
kết quả: 26


In [14]:
def trace_dp(colors, neededTime):
    dp = {None: 0}
    for x, t in zip(colors, neededTime):
        new = {c: v + t for c, v in dp.items()}
        keep = min((v for c, v in dp.items() if c != x), default=float("inf"))
        if keep < new.get(x, float("inf")):
            new[x] = keep
        dp = new
        print(f"quả màu {x} (t={t}): dp = " + ", ".join(f"{c}:{v}" for c, v in dp.items()))
    return min(dp.values())

print("kết quả:", trace_dp("aabaa", [1, 2, 3, 4, 1]))

quả màu a (t=1): dp = None:1, a:0
quả màu a (t=2): dp = None:3, a:1
quả màu b (t=3): dp = None:6, a:4, b:1
quả màu a (t=4): dp = None:10, a:1, b:5
quả màu a (t=1): dp = None:11, a:2, b:6
kết quả: 2


## 9. Code nộp LeetCode

In [15]:
class Solution:
    def minCost(self, colors: str, neededTime: List[int]) -> int:
        ans, cur_max = 0, neededTime[0]
        for i in range(1, len(colors)):
            t = neededTime[i]
            if colors[i] == colors[i - 1]:
                ans += min(cur_max, t)
                cur_max = max(cur_max, t)
            else:
                cur_max = t
        return ans

sol = Solution()
for colors, t, want in examples:
    print(colors, t, "->", sol.minCost(colors, t), "| mong đợi", want)

abaac [1, 2, 3, 4, 5] -> 3 | mong đợi 3
abc [1, 2, 3] -> 0 | mong đợi 0
aabaa [1, 2, 3, 4, 1] -> 2 | mong đợi 2


## 10. Vì sao thuật toán đúng

**Cận dưới.** Xét một nhóm liên tiếp cùng màu `G` (tối đa: hai đầu là biên dây hoặc quả khác màu). Trong lời giải bất kỳ, nhóm `G` còn lại **tối đa 1** quả: nếu còn 2 quả thì giữa chúng (trong dãy gốc) chỉ có quả cùng màu đã bị gỡ ⇒ sau khi dồn, chúng liền nhau — vi phạm. Vậy chi phí trong `G` ít nhất là `tổng(G) − max(G)`. Cộng lại qua các nhóm (rời nhau) ⇒ mọi lời giải tốn ít nhất `Σ (tổng − max)`.

**Đạt được.** Giữ đúng quả đắt nhất của mỗi nhóm. Hai quả được giữ liên tiếp thuộc hai nhóm **kề nhau** trong dãy gốc, mà hai nhóm kề nhau khác màu ⇒ hợp lệ. Chi phí đúng bằng cận dưới ⇒ tối ưu.

**Cách 4 = cách 3.** Bất biến: tại cuối mỗi bước trong một nhóm, `cur_max` = max các phần tử đã xét của nhóm và `ans` đã cộng `tổng − max` của phần đã xét. Bước mới với `t`: tổng tăng `t`, max mới `= max(cur_max, t)` ⇒ `tổng − max` tăng đúng `t + cur_max − max(cur_max, t) = min(cur_max, t)`.

**Cách 2 (DP).** Quyết định giữ quả `i` chỉ hợp lệ khi quả được giữ gần nhất khác màu — đó chính là thông tin trong trạng thái; mọi chuỗi quyết định hợp lệ được biểu diễn, nên `min` cuối cùng là tối ưu.

## 11. Phân tích Time / Space

Gọi `n = len(colors)`, `Σ = 26` màu.

| Cách | Time | Space phụ |
|---|---|---|
| 1. Brute force bitmask | `O(2ⁿ · n)` | `O(n)` |
| 2. DP theo màu cuối | `O(n · Σ)` | `O(Σ)` |
| 3. Nhóm `tổng − max` | `O(n)` | `O(1)` (bản `groupby` tạo list nhóm tạm) |
| 4. Một lượt tham lam | `O(n)` | `O(1)` |

Không có khác biệt average/worst cho các cách 2–4.

## 12. Bộ test

In [16]:
ALL = [brute_force, min_cost_dp, min_cost_groups, min_cost_groupby, min_cost_greedy, Solution().minCost]

for f in ALL:
    for colors, t, want in examples:
        assert f(colors, list(t)) == want, (f.__name__, colors)
print("Ví dụ đề: OK")

edge = [
    ("a", [5], 0),                                   # n = 1
    ("aa", [3, 3], 3),                               # hai quả bằng nhau
    ("aaaa", [1, 1, 1, 1], 3),                       # toàn cùng màu
    ("abab", [9, 9, 9, 9], 0),                       # xen kẽ
    ("aaabbbabbbb", [3, 5, 10, 7, 5, 3, 5, 5, 4, 8, 1], 26),
    ("bbbaaa", [4, 9, 3, 8, 8, 9], 23),
]
for f in ALL:
    for colors, t, want in edge:
        assert f(colors, list(t)) == want, (f.__name__, colors, f(colors, list(t)), want)
print("Edge case: OK")

Ví dụ đề: OK
Edge case: OK


In [17]:
# Ngẫu nhiên seed cố định, đối chiếu với brute force (n nhỏ)
rng = random.Random(1578)
for _ in range(1200):
    n = rng.randint(1, 11)
    alphabet = rng.choice(["a", "ab", "abc", "abcdz"])
    colors = "".join(rng.choice(alphabet) for _ in range(n))
    t = [rng.randint(1, rng.choice([3, 10**4])) for _ in range(n)]
    want = brute_force(colors, t)
    for f in ALL[1:]:
        assert f(colors, list(t)) == want, (f.__name__, colors, t)
print("1200 bộ ngẫu nhiên: OK")

1200 bộ ngẫu nhiên: OK


In [18]:
# Test lớn đúng ràng buộc n = 10^5
rng = random.Random(77)
big = [
    ("a" * 10**5, [10**4] * 10**5),
    ("".join(rng.choice("ab") for _ in range(10**5)), [rng.randint(1, 10**4) for _ in range(10**5)]),
    ("".join(rng.choice("abcdefghijklmnopqrstuvwxyz") for _ in range(10**5)), [rng.randint(1, 10**4) for _ in range(10**5)]),
]
for colors, t in big:
    res = []
    for f in [min_cost_dp, min_cost_groups, min_cost_greedy]:
        t0 = time.perf_counter()
        res.append(f(colors, t))
        print(f"  {f.__name__:<16} {res[-1]:>11} {time.perf_counter() - t0:.3f}s")
    assert len(set(res)) == 1
assert min_cost_greedy("a" * 10**5, [10**4] * 10**5) == (10**5 - 1) * 10**4
print("Test lớn: OK")

  min_cost_dp        999990000 0.070s


  min_cost_groups    999990000 0.010s
  min_cost_greedy    999990000 0.011s
  min_cost_dp        194947552 0.087s
  min_cost_groups    194947552 0.015s
  min_cost_greedy    194947552 0.009s


  min_cost_dp         12713350 0.406s
  min_cost_groups     12713350 0.017s
  min_cost_greedy     12713350 0.006s
Test lớn: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Chỉ so sánh cặp kề nhau, không mang theo `max`

Cộng `min(t[i-1], t[i])` cho mọi cặp cùng màu kề nhau là sai: quả rẻ ở giữa có thể bị "gỡ hai lần" trong khi quả đắt bên cạnh bị bỏ sót.

In [19]:
def wrong_pairwise(colors, neededTime):
    ans = 0
    for i in range(1, len(colors)):
        if colors[i] == colors[i - 1]:
            ans += min(neededTime[i - 1], neededTime[i])     # SAI: không cập nhật quả được giữ
    return ans

colors, t = "aaa", [3, 1, 2]
print("sai:", wrong_pairwise(colors, t), "| đúng:", Solution().minCost(colors, t))
# Bản sai gỡ quả 1 hai lần (1 + 1) và giữ cả 3 lẫn 2 -> vẫn còn "aa".

sai: 2 | đúng: 3


### Lỗi 2 — Gom theo màu **toàn cục** thay vì theo nhóm liên tiếp

Hai quả cùng màu nhưng bị ngăn bởi quả khác màu không xung đột.

In [20]:
from collections import defaultdict

def wrong_global_color(colors, neededTime):
    by_color = defaultdict(list)
    for c, t in zip(colors, neededTime):
        by_color[c].append(t)
    return sum(sum(ts) - max(ts) for ts in by_color.values())   # SAI

colors, t = "aba", [1, 2, 3]
print("sai:", wrong_global_color(colors, t), "| đúng:", Solution().minCost(colors, t))

sai: 1 | đúng: 0


### Lỗi 3 — Tự tách nhóm nhưng quên xử lý **nhóm cuối cùng**

Kiểu viết "khi màu đổi thì chốt nhóm cũ" sẽ không chốt nhóm cuối khi vòng lặp kết thúc.

In [21]:
def wrong_forget_last(colors, neededTime):
    ans, s, mx = 0, neededTime[0], neededTime[0]
    for i in range(1, len(colors)):
        if colors[i] == colors[i - 1]:
            s += neededTime[i]
            mx = max(mx, neededTime[i])
        else:
            ans += s - mx                                   # chốt nhóm cũ
            s = mx = neededTime[i]
    return ans                                              # SAI: thiếu ans += s - mx

for colors, t in [("abcc", [1, 2, 3, 4]), ("aabaa", [1, 2, 3, 4, 1])]:
    print(colors, "-> sai:", wrong_forget_last(colors, t), "| đúng:", Solution().minCost(colors, t))

abcc -> sai: 0 | đúng: 3
aabaa -> sai: 1 | đúng: 2


### Lỗi 4 — Cách 4 nhưng quên reset `cur_max` khi sang nhóm mới

`cur_max` của nhóm cũ bị mang sang nhóm mới ⇒ "giữ" một quả không thuộc nhóm hiện tại.

In [22]:
def wrong_no_reset(colors, neededTime):
    ans, cur_max = 0, neededTime[0]
    for i in range(1, len(colors)):
        t = neededTime[i]
        if colors[i] == colors[i - 1]:
            ans += min(cur_max, t)
        cur_max = max(cur_max, t)                           # SAI: không reset khi đổi màu
    return ans

colors, t = "abb", [10, 2, 3]
print("sai:", wrong_no_reset(colors, t), "| đúng:", Solution().minCost(colors, t))

sai: 3 | đúng: 2


### Lỗi 5 — Sửa trực tiếp `neededTime` (side effect) rồi dùng lại mảng

Một số lời giải ghi đè `neededTime[i] = max(...)` để "mang theo max". Đúng với một lần gọi, nhưng nếu mảng được dùng lại (ví dụ trong test hoặc khi gọi hai hàm liên tiếp) thì kết quả lần sau sai.

In [23]:
def in_place_version(colors, neededTime):
    ans = 0
    for i in range(1, len(colors)):
        if colors[i] == colors[i - 1]:
            ans += min(neededTime[i - 1], neededTime[i])
            neededTime[i] = max(neededTime[i - 1], neededTime[i])   # ghi đè dữ liệu đầu vào
    return ans

colors, t = "aaa", [3, 1, 2]
first = in_place_version(colors, t)
print("lần 1:", first, "| mảng sau khi gọi:", t)
print("dùng lại mảng đã bị sửa cho hàm khác -> sai:", Solution().minCost(colors, t), "| đúng:", Solution().minCost(colors, [3, 1, 2]))

lần 1: 3 | mảng sau khi gọi: [3, 3, 3]
dùng lại mảng đã bị sửa cho hàm khác -> sai: 6 | đúng: 3


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **"Không có hai phần tử kề nhau giống nhau" + chi phí xoá** ⇒ nghĩ ngay tới **nhóm liên tiếp** (run-length): mỗi nhóm giữ lại phần tử "đáng giá" nhất.
- Mẫu một lượt: `if same_as_prev: ans += min(keep, cur); keep = max(keep, cur) else: keep = cur`.
- Khi không chắc tham lam đúng ⇒ viết DP trạng thái nhỏ (màu cuối được giữ) và brute force bitmask để đối chiếu.
- Kiểm tra nhanh: `"aaa",[3,1,2]` → `3`; `"aba",[1,2,3]` → `0`; `"abaac"` → `3`.

## 15. Biến thể của bài

1. **Không có `k` phần tử liên tiếp cùng màu** (thay vì 2): mỗi nhóm giữ `k-1` phần tử đắt nhất (bài tập 5).
2. **Dây vòng** (quả đầu và quả cuối kề nhau): xoay để bắt đầu tại ranh giới đổi màu, rồi giải như dây thẳng; nếu toàn một màu thì `tổng − max` (bài tập 4).
3. **LC 1957 — Delete Characters to Make Fancy String:** không có 3 ký tự liên tiếp giống nhau (không có chi phí).
4. **LC 1047 / 1209 — Remove Adjacent Duplicates:** dùng stack, xoá dây chuyền.
5. **Trả về chính các quả bị gỡ** (bài tập 3).
6. **LC 2268/ Paint House (LC 256):** DP "màu của phần tử trước" — cùng khung trạng thái như cách 2.

## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Brute force | thử mọi tập bóng bị gỡ | `O(2ⁿ·n)` | `O(n)` | kiểm thử, `n ≤ 16` |
| 2. DP màu cuối | `dp[c]` = chi phí min, quả giữ cuối màu `c` | `O(n·Σ)` | `O(Σ)` | biến thể có ràng buộc phức tạp, khi chưa thấy insight |
| 3. Nhóm `tổng − max` | tách nhóm liên tiếp, giữ quả đắt nhất | `O(n)` | `O(1)` | dễ giải thích, dễ chứng minh |
| 4. Một lượt tham lam | cộng `min(cur_max, t)`, cập nhật `cur_max` | `O(n)` | `O(1)` | code ngắn nhất, phỏng vấn |

## 17. 5 bài tập vận dụng

**Bài 1 (đoán output):** `Solution().minCost("bbbaaa", [4, 9, 3, 8, 8, 9])` trả về bao nhiêu?

**Bài 2 (tính tay):** Với `colors = "abbbcc"`, `neededTime = [1, 2, 3, 4, 5, 6]`, liệt kê các nhóm, quả được giữ ở mỗi nhóm và tổng chi phí.

**Bài 3 (điền TODO):** Viết `ex3_removed_indices(colors, neededTime)` trả về danh sách chỉ số các quả bị gỡ trong **một** lời giải tối ưu.

**Bài 4 (tự viết lại — dây vòng):** Viết `ex4_min_cost_circular(colors, neededTime)` khi quả cuối và quả đầu **cũng kề nhau**. Quy ước: nếu chỉ còn 1 quả thì hợp lệ.

**Bài 5 (mini challenge):** Viết `ex5_min_cost_k(colors, neededTime, k)`: chi phí nhỏ nhất để **không có `k` quả liên tiếp cùng màu** (`k ≥ 2`).

In [24]:
# Bài 3 — khung TODO (chỉ định nghĩa, KHÔNG gọi)
def ex3_removed_indices(colors, neededTime):
    removed = []
    keep = 0          # chỉ số quả đang được giữ trong nhóm hiện tại
    for i in range(1, len(colors)):
        # TODO: nếu cùng màu với quả trước: so sánh neededTime[keep] và neededTime[i],
        #       thêm chỉ số rẻ hơn vào removed, cập nhật keep
        # TODO: nếu khác màu: keep = i
        pass
    return removed

In [25]:
# Bài 4 — khung TODO (chỉ định nghĩa, KHÔNG gọi)
def ex4_min_cost_circular(colors, neededTime):
    n = len(colors)
    # TODO: nếu mọi màu giống nhau -> sum - max
    # TODO: tìm vị trí s mà colors[s] != colors[s-1]; xoay chuỗi để bắt đầu tại s
    # TODO: giải như dây thẳng trên chuỗi đã xoay
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Hai nhóm: `bbb (4,9,3)` và `aaa (8,8,9)`. (2) Mỗi nhóm `tổng − max`. (3) `(16 − 9) + (25 − 9)`.

**Bài 2.** (1) Nhóm: `a(1)`, `bbb(2,3,4)`, `cc(5,6)`. (2) Giữ `1`, `4`, `6`. (3) Chi phí `0 + 5 + 5`.

**Bài 3.** (1) Theo dõi **chỉ số** quả đắt nhất thay vì giá trị. (2) Nếu `neededTime[i] > neededTime[keep]` thì gỡ `keep` và `keep = i`, ngược lại gỡ `i`. (3) Kiểm tra: tổng thời gian các chỉ số bị gỡ = đáp án, và dãy còn lại hợp lệ (kể cả xét vòng nếu cần).

**Bài 4.** (1) Nếu toàn một màu: chỉ giữ 1 quả. (2) Ngược lại, tồn tại `s` với `colors[s] != colors[s-1]` (chỉ số vòng), nên điểm cắt giữa `s-1` và `s` không nằm trong nhóm nào. (3) `colors[s:] + colors[:s]` rồi gọi lời giải dây thẳng.

**Bài 5.** (1) Vẫn tách nhóm liên tiếp. (2) Trong nhóm, được giữ tối đa `k-1` quả (giữ bất kỳ `k-1` quả nào cũng không tạo `k` quả liền nhau, vì hai đầu nhóm là màu khác). (3) Giữ `k-1` quả đắt nhất: `heapq.nlargest(k-1, ts)`.

## 19. Đáp án tham khảo cho bài tập

In [26]:
# Bài 1, 2
ans1 = Solution().minCost("bbbaaa", [4, 9, 3, 8, 8, 9])
print("Bài 1:", ans1)
assert ans1 == (16 - 9) + (25 - 9) == 23

ans2 = Solution().minCost("abbbcc", [1, 2, 3, 4, 5, 6])
print("Bài 2:", ans2)
show_groups("abbbcc", [1, 2, 3, 4, 5, 6])
assert ans2 == 10
print("Bài 1, 2: OK")

Bài 1: 23
Bài 2: 10
  nhóm a     thời gian [1]: giữ 1, gỡ tốn 0
  nhóm bbb   thời gian [2, 3, 4]: giữ 4, gỡ tốn 5
  nhóm cc    thời gian [5, 6]: giữ 6, gỡ tốn 5
  tổng = 10
Bài 1, 2: OK


In [27]:
# Bài 3
def ex3_removed_indices_sol(colors, neededTime):
    removed, keep = [], 0
    for i in range(1, len(colors)):
        if colors[i] == colors[i - 1]:
            if neededTime[i] > neededTime[keep]:
                removed.append(keep)
                keep = i
            else:
                removed.append(i)
        else:
            keep = i
    return sorted(removed)

rng = random.Random(3)
for _ in range(1000):
    n = rng.randint(1, 12)
    colors = "".join(rng.choice("abc") for _ in range(n))
    t = [rng.randint(1, 6) for _ in range(n)]
    rem = ex3_removed_indices_sol(colors, t)
    kept = [colors[i] for i in range(n) if i not in set(rem)]
    assert all(a != b for a, b in zip(kept, kept[1:]))
    assert sum(t[i] for i in rem) == Solution().minCost(colors, t)
print("Bài 3: OK, ví dụ:", ex3_removed_indices_sol("aabaa", [1, 2, 3, 4, 1]))

Bài 3: OK, ví dụ: [0, 4]


In [28]:
# Bài 4 — dây vòng
def ex4_min_cost_circular_sol(colors, neededTime):
    n = len(colors)
    if all(c == colors[0] for c in colors):
        return sum(neededTime) - max(neededTime)
    s = next(i for i in range(n) if colors[i] != colors[i - 1])   # i-1 = -1 là quả cuối
    rc = colors[s:] + colors[:s]
    rt = neededTime[s:] + neededTime[:s]
    return Solution().minCost(rc, rt)

def ex4_brute(colors, neededTime):
    n, best = len(colors), sum(neededTime)
    for mask in range(1 << n):
        kept = [colors[i] for i in range(n) if not mask >> i & 1]
        ok = all(kept[i] != kept[(i + 1) % len(kept)] for i in range(len(kept))) if len(kept) > 1 else True
        if ok:
            best = min(best, sum(neededTime[i] for i in range(n) if mask >> i & 1))
    return best

print("Bài 4:", ex4_min_cost_circular_sol("abaa", [1, 2, 3, 4]), ex4_min_cost_circular_sol("aaa", [1, 2, 3]))
rng = random.Random(4)
for _ in range(800):
    n = rng.randint(1, 10)
    colors = "".join(rng.choice("abc") for _ in range(n))
    t = [rng.randint(1, 8) for _ in range(n)]
    assert ex4_min_cost_circular_sol(colors, t) == ex4_brute(colors, t), (colors, t)
print("Bài 4: OK")

Bài 4: 4 3


Bài 4: OK


In [29]:
# Bài 5 — không có k quả liên tiếp cùng màu
import heapq

def ex5_min_cost_k(colors, neededTime, k):
    ans = 0
    for _, grp in groupby(zip(colors, neededTime), key=lambda p: p[0]):
        ts = [t for _, t in grp]
        if len(ts) >= k:
            ans += sum(ts) - sum(heapq.nlargest(k - 1, ts))
    return ans

def ex5_brute(colors, neededTime, k):
    n, best = len(colors), sum(neededTime)
    for mask in range(1 << n):
        kept = [colors[i] for i in range(n) if not mask >> i & 1]
        ok = all(len(set(kept[i:i + k])) > 1 for i in range(len(kept) - k + 1))
        if ok:
            best = min(best, sum(neededTime[i] for i in range(n) if mask >> i & 1))
    return best

print("Bài 5:", ex5_min_cost_k("aaaa", [1, 2, 3, 4], 3), ex5_min_cost_k("aaabbbabbbb", [3, 5, 10, 7, 5, 3, 5, 5, 4, 8, 1], 2))
assert ex5_min_cost_k("aaaa", [1, 2, 3, 4], 3) == 3
rng = random.Random(5)
for _ in range(600):
    n = rng.randint(1, 10)
    colors = "".join(rng.choice("ab") for _ in range(n))
    t = [rng.randint(1, 9) for _ in range(n)]
    k = rng.randint(2, 4)
    assert ex5_min_cost_k(colors, t, k) == ex5_brute(colors, t, k), (colors, t, k)
print("Bài 5: OK")

Bài 5: 3 26


Bài 5: OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
class Solution:
    def minCost(self, colors, neededTime):
        ans, cur_max = 0, neededTime[0]
        for i in range(1, len(colors)):
            t = neededTime[i]
            if colors[i] == colors[i - 1]:     # cùng nhóm: gỡ quả rẻ hơn
                ans += min(cur_max, t)
                cur_max = max(cur_max, t)      # giữ quả đắt hơn
            else:                              # nhóm mới
                cur_max = t
        return ans
```

**Template "nhóm liên tiếp":** mỗi nhóm cùng giá trị liền nhau → giữ phần tử tốt nhất (hoặc `k-1` phần tử tốt nhất) → chi phí `tổng − phần giữ`.

**Checklist trước khi submit:**
- [ ] Nhóm theo **liên tiếp**, không theo màu toàn cục.
- [ ] Mang theo `cur_max` (quả được giữ), không chỉ so cặp kề nhau.
- [ ] Reset `cur_max = t` khi đổi màu.
- [ ] Nếu tự tách nhóm: nhớ chốt nhóm cuối.
- [ ] Không sửa `neededTime` tại chỗ nếu còn dùng lại.
- [ ] Thử nhanh: ví dụ → `3, 0, 2`; `"aaa",[3,1,2]` → `3`; `"a",[5]` → `0`.